In [9]:
import os
import openai

# 🔥 Clear Azure settings
os.environ.pop("OPENAI_API_TYPE", None)
os.environ.pop("OPENAI_API_BASE", None)
os.environ.pop("OPENAI_API_VERSION", None)

# 🔥 Set correct OpenAI config
openai.api_type = "open_ai"
openai.api_base = "https://api.openai.com/v1"
openai.api_version = None

openai.api_key = os.getenv("OPENAI_API_KEY")


StatementMeta(, f438b1f1-51d8-4b48-b9b5-78de16e36d54, 21, Finished, Available, Finished, False)

Building the Master Analytics Table...


AnalysisException: [UNRESOLVED_USING_COLUMN_FOR_JOIN] USING column `GeographyKey` cannot be resolved on the left side of the join. The left-side columns: [`ArabicDescription`, `CalendarQuarter`, `CalendarSemester`, `CalendarYear`, `CarrierTrackingNumber`, `ChineseDescription`, `Class`, `Color`, `CurrencyKey`, `CustomerKey`, `CustomerPONumber`, `DateKey`, `DayNumberOfMonth`, `DayNumberOfWeek`, `DayNumberOfYear`, `DaysToManufacture`, `DealerPrice`, `DiscountAmount`, `DueDate`, `DueDateKey`, `EmployeeKey`, `EndDate`, `EnglishDayNameOfWeek`, `EnglishDescription`, `EnglishMonthName`, `EnglishProductName`, `ExtendedAmount`, `FinishedGoodsFlag`, `FiscalQuarter`, `FiscalSemester`, `FiscalYear`, `Freight`, `FrenchDayNameOfWeek`, `FrenchDescription`, `FrenchMonthName`, `FrenchProductName`, `FullDateAlternateKey`, `GermanDescription`, `HebrewDescription`, `JapaneseDescription`, `LargePhoto`, `ListPrice`, `ModelName`, `MonthNumberOfYear`, `OrderDate`, `OrderDateKey`, `OrderQuantity`, `ProductAlter

In [8]:
SYSTEM_PROMPT = """
You are a data analyst.

You are working with table: fact_sales_combined_tbl, dimProduct, dimCustomer,dimdate,dimgeography,dimproductcategory,dimproductsubcategory,dimseller,simsalesterritory, factinternetsales,factreseller

Columns:
- SalesType (Internet/Reseller)
- SalesAmount
- OrderQuantity
- ProductKey
- OrderDateKey
- CustomerKey
- ResellerKey
- EnglishProductName

Rules:
- Generate ONLY Spark SQL query
- Always use SUM(SalesAmount) for revenue
- Use GROUP BY when needed
- Limit results to 100 rows
- Do not explain anything
"""

StatementMeta(, f438b1f1-51d8-4b48-b9b5-78de16e36d54, 20, Finished, Available, Finished, False)

--- AGENT QUERY ---
SELECT 
    Country, 
    ProductCategoryName, 
    SUM(SalesAmount) AS TotalSales, 
    ((SUM(SalesAmount) - LAG(SUM(SalesAmount)) OVER (PARTITION BY Country, ProductCategoryName ORDER BY CalendarYear)) / LAG(SUM(SalesAmount)) OVER (PARTITION BY Country, ProductCategoryName ORDER BY CalendarYear)) AS YoYGrowth
FROM 
    fact_master_analytics
WHERE 
    EnglishProductName NOT LIKE '%?%'
GROUP BY 
    Country, 
    ProductCategoryName, 
    CalendarYear
ORDER BY 
    Country, 
    ProductCategoryName, 
    CalendarYear
LIMIT 100

Execution Error: [TABLE_OR_VIEW_NOT_FOUND] The table or view `fact_master_analytics` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 7 pos 4;
'GlobalLimit 100
+- 'LocalLimit 100
   +- 'Sor

In [3]:
def generate_sql(question):
    response = openai.ChatCompletion.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question}
        ]
    )
    
    return response["choices"][0]["message"]["content"].strip()

StatementMeta(, f438b1f1-51d8-4b48-b9b5-78de16e36d54, 10, Finished, Available, Finished, False)

ImportError: cannot import name 'OpenAI' from 'openai' (/nfs4/pyenv-7968602d-963b-44d0-9367-64914ac3851b/lib/python3.11/site-packages/openai/__init__.py)

In [ ]:
def is_safe(sql):
    sql = sql.lower()
    forbidden = ["drop", "delete", "truncate", "update", "insert"]
    return not any(word in sql for word in forbidden)

In [ ]:
def run_query(sql):
    return spark.sql(sql)

In [ ]:
def clean_sql(sql):
    sql = sql.strip()
    
    # remove markdown blocks
    if sql.startswith("```"):
        sql = sql.replace("```sql", "").replace("```", "")
    
    return sql.strip()

In [ ]:
question = "What is total sales amount for product [Touring Tire] for all years of orderdatekey"

sql = generate_sql(question)

# 🔥 Clean the SQL
sql = clean_sql(sql)

print("Generated SQL:\n", sql)

if is_safe(sql):
    result = spark.sql(sql)
    display(result)
else:
    print("Unsafe query blocked")